<a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/wl-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/wl-classification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab image-classification notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> is an open-source PyTorch tool for dataset debugging, mislabel detection, and mid-training data curation. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details, and raise an issue on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a> for support.</div>

# Image Classification with WeightsLab

This notebook trains a small CNN on **MNIST** and instruments it with WeightsLab so every training signal is traced **back to the exact samples** producing it.

Most data problems (mislabels, outliers, class imbalance) stay invisible until your model tells you through the loss. By wrapping your training objects with `wl.watch_or_edit(...)`, WeightsLab records **per-sample** loss and metrics live, so you can rank the worst samples, spot bad labels, and curate the dataset **without restarting training**.

### What you'll do
1. Install WeightsLab.
2. Set every knob in one **config** dict (like a `config.yaml`).
3. Wrap the model, optimizer, dataloaders, loss, and metric with the SDK.
4. Train while per-sample signals are captured.
5. Rank the highest-loss samples inline, and (optionally) open the live **Weights Studio** UI.

## Setup

Install WeightsLab from PyPI. On Colab the free **T4 GPU** runtime is plenty for this demo (`Runtime -> Change runtime type -> T4 GPU`).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [ ]:
# Install WeightsLab. Colab already ships torch, torchvision, numpy, scikit-learn
# and Pillow, so nothing extra is needed here.
%pip install weightslab

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase.

In [ ]:
import os
import tempfile
import logging
import itertools
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy
from tqdm.auto import tqdm

import weightslab as wl
from weightslab.examples.utils.baseline_models.pytorch.models import FashionCNN as CNN

logging.basicConfig(level=logging.ERROR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. A dataset that carries sample identity

WeightsLab attributes every signal to a **sample id**. This thin wrapper around torchvision's MNIST returns `(image, id, label)` and attaches a virtual filepath per sample, so signals can be traced back to individual images in the UI.

In [ ]:
class MNISTCustomDataset(Dataset):
    """MNIST that returns (image, index, label) and tracks a virtual filepath."""

    def __init__(self, root, train=True, download=False, transform=None):
        self.mnist = datasets.MNIST(root=root, train=train, download=download, transform=None)
        self.transform = transform
        self.train = train
        split = "train" if train else "test"
        self.filepaths = {}
        for idx in range(len(self.mnist)):
            label = self.mnist.targets[idx].item()
            self.filepaths[idx] = os.path.join(
                "MNIST", "processed", split, f"class_{label}", f"sample_{idx:05d}.pt"
            )

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        if self.transform:
            image = self.transform(image)
        return image, idx, label

## 3. Configuration

Every tunable lives here, in one dict, like a `config.yaml` with comments. Wrapping it with `flag="hyperparameters"` lets the Studio UI read (and live-edit) these values while training.

In [ ]:
log_dir = tempfile.mkdtemp(prefix="weightslab_mnist_")

config = {
    # -- Experiment -------------------------------------------------------
    "experiment_name": "mnist_classification",    # name shown in Weights Studio
    "device": str(device),                        # "cuda" if a GPU is available, else "cpu"
    "root_log_dir": log_dir,                      # where signal history / dataframes are written
    "num_classes": 10,                            # MNIST digits 0-9

    # -- Training schedule ------------------------------------------------
    "training_steps_to_do": 8000,                 # total optimizer steps (raise for a longer live run)
    "eval_full_to_train_steps_ratio": 500,        # run a full eval every N steps
    "experiment_dump_to_train_steps_ratio": 250,   # dump model weights ratio
    "write_export_ratio": 1000,                   # export signal history + dataframe every N steps

    # Configure global dataframe storage
    "ledger_enable_flushing_threads": True,
    "ledger_enable_h5_persistence": True,
    "ledger_flush_max_rows": 1024,
    "ledger_flush_interval": 20.0,

    # -- Optimizer --------------------------------------------------------
    "learning_rate": 0.001,                        # Adam learning rate

    # -- Data loaders -----------------------------------------------------
    # One block per loader, keyed by loader_name. EVERY kwarg passed to
    # wl.watch_or_edit(..., flag="data") lives here so the wrap cell below stays
    # declarative — no dataloader settings hardcoded in the cells. Add a new
    # loader by adding another block here.
    "data": {
        "train_loader": {
            "batch_size": 128,                     # training batch size
            "shuffle": True,                      # shuffle each epoch
            "is_training": True,                  # marks this loader as the training split
            "compute_hash": False,                # skip per-sample content hashing (faster init)
            "preload_labels": True,               # preload labels into the ledger at startup
            "preload_metadata": False,            # load metadata lazily on first access
            "enable_h5_persistence": True,        # persist per-sample stats to the H5 store
        },
        "test_loader": {
            "batch_size": 128,                    # evaluation batch size
            "shuffle": False,                     # keep test order stable
            "is_training": False,                 # marks this loader as an eval split
            "compute_hash": False,
            "preload_labels": True,
            "preload_metadata": False,
            "enable_h5_persistence": True,
        },
    },

    # -- Services ---------------------------------------------------------
    "serving_grpc": True,                         # expose the gRPC backend for Weights Studio
    "serving_bore": True,                         # expose the bore tunnel for Weights Studio
}

wl.watch_or_edit(config, flag="hyperparameters", poll_interval=1.0)
print("Experiment logs ->", log_dir)

## 4. Wrap the training objects

This is the heart of WeightsLab. Each object is passed through `wl.watch_or_edit(...)` with a `flag` describing its role. The returned objects behave exactly like the originals, but now report their state and per-sample signals to WeightsLab.

In [ ]:
data_root = os.path.join(log_dir, "data")
os.makedirs(data_root, exist_ok=True)

train_ds = MNISTCustomDataset(root=data_root, train=True, download=True,
                              transform=transforms.ToTensor())
test_ds = MNISTCustomDataset(root=data_root, train=False, download=True,
                             transform=transforms.ToTensor())

# Model + optimizer
model = wl.watch_or_edit(CNN().to(device), flag="model", device=device, compute_dependencies=True)
optimizer = wl.watch_or_edit(
    optim.Adam(model.parameters(), lr=config["learning_rate"]), flag="optimizer")

# Tracked dataloaders — all loader settings come from config["data"][<loader_name>],
# so nothing is hardcoded here. Unpack each block straight into the wrapper.
train_loader = wl.watch_or_edit(
    train_ds, flag="data", loader_name="train_loader",
    **config["data"]["train_loader"],
)
test_loader = wl.watch_or_edit(
    test_ds, flag="data", loader_name="test_loader",
    **config["data"]["test_loader"],
)

# Watched losses + metric (they log themselves per sample)
train_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"),
                                   flag="loss", signal_name="train-loss-CE", log=True)
test_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"),
                                  flag="loss", signal_name="test-loss-CE", log=True)
metric = wl.watch_or_edit(
    Accuracy(task="multiclass", num_classes=config["num_classes"]).to(device),
    flag="metric", signal_name="metric-ACC", log=True)

## 5. Train and evaluate steps

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab which phase it's in. `criterion(..., batch_ids=ids, preds=preds)` passes the sample ids so the loss is stored **per sample**, and `wl.save_signals(...)` logs a custom per-sample accuracy signal during evaluation.

In [ ]:
def train(loader, model, optimizer, criterion, device):
    with wl.guard_training_context:
        inputs, ids, labels = next(loader)
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        preds = logits.argmax(dim=1, keepdim=True)

        loss = criterion(logits.float(), labels.long(), batch_ids=ids, preds=preds)
        total_loss = loss.mean()
        total_loss.backward()
        optimizer.step()
    return total_loss.detach().cpu().item()


import time
def test(loader, model, criterion, metric, device, n_batches):
    losses = torch.tensor(0.0, device=device)
    st = time.time()
    for inputs, ids, labels in loader:
        with wl.guard_testing_context:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=1, keepdim=True)

            loss = criterion(logits, labels, batch_ids=ids, preds=preds)
            losses += loss.mean()
            metric.update(logits, labels)

            correct = (preds.view(-1) == labels.view(-1)).float()
            wl.save_signals(
                preds_raw=logits, targets=labels, batch_ids=ids, preds=preds,
                signals={
                    "test_metric/Accuracy_per_sample": correct,
                    "test_metric/Inverse_Accuracy_per_sample": 1.0 - correct,
                },
            )
    print(f"Compute test in {time.time()-st}s")
    return (losses / n_batches).item(), (metric.compute() * 100).item()

## 6. Serve and train

`wl.serve(serving_grpc=True)` starts the background gRPC server (non-blocking) that Weights Studio connects to. `wl.start_training(...)` flips the experiment into the *training* state, then we run the loop, periodically evaluating and exporting signals.

In [ ]:
wl.serve(serving_grpc=True, serving_bore=True)

In [ ]:
import itertools

wl.start_training(timeout=3)

eval_ratio = config["eval_full_to_train_steps_ratio"]
export_ratio = config["write_export_ratio"]
n_test_batches = len(test_loader)

test_loss = test_acc = None
pbar = tqdm(itertools.count(), desc="Training")
# Open-ended: the loop has no step budget, so training runs until you
# interrupt the kernel (or pause it from the studio). The exports sit in
# `finally` for exactly that reason -- interrupting is the normal way to stop
# here, and it would otherwise skip them.
try:
    for step in pbar:
        age = model.get_age() if hasattr(model, "get_age") else step

        train_loss = train(train_loader, model, optimizer, train_criterion, device)

        if age == 0 and age % eval_ratio == 0:
            test_loss, test_acc = test(test_loader, model, test_criterion, metric, device, n_test_batches)

        postfix = {"loss": f"{train_loss:.3f}"}
        if test_acc is not None:
            postfix["test_acc"] = f"{test_acc:.1f}%"
        pbar.set_postfix(postfix)
finally:
    wl.write_history()
    wl.write_dataframe()
    print("Training complete. Logs at:", log_dir)

## 9. See it live in Weights Studio

Everything above ran headless. The real payoff is the **Weights Studio** UI, where you browse the highest-loss images, filter by class, and curate the dataset mid-training.

Studio runs as a local Docker stack (a static frontend + an Envoy gRPC-Web proxy). **Colab has no Docker daemon**, so you don't run the UI *inside* Colab - you run Studio on your own machine and point it at this notebook's backend using the `bore.pub:<port>` endpoint **printed in Section 6**.

**On your machine** (with Docker Desktop):
```bash
pip install weightslab

# Terminal 1 - launch the UI (plaintext HTTP, the default)
weightslab start                       # opens http://localhost:5173

# Terminal 2 - bridge the Colab backend to localhost:50051
weightslab tunnel bore.pub:12345           # in another window, the host:port printed in Section 6
```

Then open **http://localhost:5173** - Studio connects through your local Envoy -> `weightslab tunnel` -> `bore.pub` -> this Colab backend, and you watch training stream live.

> Note: keep it plaintext end-to-end (the default `weightslab start`, raw-TCP `bore`) so gRPC's HTTP/2 frames pass through untouched.

Prefer to keep it all local? Run this same example on your own machine (`weightslab start example --cls`) and launch the UI next to it - no tunnel required.

## 10. Curate in the UI to boost performance

This is where WeightsLab pays off: instead of training longer on all 60k samples, use the live signals to **find the samples that matter, keep only those, and fine-tune**. Everything below happens in Weights Studio while the backend keeps running — no code changes, no restart.

> **Baseline before curating:** the evaluation metric (test accuracy) sits around **0.86**.

**1. See more of the dataset.** In the **grid explorer**, increase the number of images per page so you can scan many samples at once.

**2. Find the easy and hard samples.** Sort the grid by **`train-loss-CE` in decreasing order**, then **generate the histogram** for that signal. The distribution is bimodal — a tail of **hard** examples (training loss **> 2.45**) and a bulk of **easy** ones (training loss **< 1.461**). These two extremes carry the signal; the vast middle is largely redundant.

**3. Let the agent tag them.** Initialize the **agent** (the panel in the UI), then ask it:

> *Tag training samples with train loss greater than 2.45 as "hard_ex" and training samples with train loss lower than 1.45 as "easy_ex".*

Inspect the new `tag:hard_ex` / `tag:easy_ex` columns in the grid to confirm the tagging looks right.

**4. Drop the redundant middle.** Once the tags look good, ask the agent:

> *Discard train samples that have no "easy_ex" and no "hard_ex" tag.*

The training set collapses from **~60k to roughly 3–4k** samples — only the informative extremes remain.

**5. Fine-tune on the curated set.** **Resume** training (from the UI, or by re-running the training cell) and let it run for about **6,500 steps** on this much smaller, higher-signal dataset.

**6. Evaluate.** Trigger a full evaluation from the UI: **right-click `resume` → `evaluate` → click `evaluate` → select `test_loader`**, then wait for it to finish.

> **Result:** test accuracy climbs to roughly **0.95–0.99**, a large jump over the 0.86 baseline — reached by training on **~15× fewer** samples. Curating *which* data the model sees beat simply training on *more* of it.

This is the core WeightsLab loop: **read the per-sample signals → curate the dataset → fine-tune → measure**, all without leaving the UI or restarting the run.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>